
# PMCMC: Particle Marginal Metropolis--Hastings (PMMH)

This notebook implements the PMMH algorithm following the provided pseudo-code:

1. At iteration \(k=0\), initialize \(\theta(0)\), run an SMC algorithm, sample a trajectory \(X_{1:T}(0)\), and compute \(\widehat p_{\theta(0)}(y_{1:T})\).
2. At iteration \(k\geq 1\), propose \(\theta'\sim q(\theta'\mid \theta(k-1))\).
3. Run SMC under \(\theta'\), sample \(X'_{1:T}\), and compute \(\widehat p_{\theta'}(y_{1:T})\).
4. Accept or reject using the PMMH ratio.

The stochastic volatility model is

\[
X_t=\rho X_{t-1}+\sigma V_t,\qquad
Y_t=\beta \exp(X_t/2) W_t.
\]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import invgamma
from scipy.special import logsumexp
from scipy.stats import gaussian_kde


## Load data

The file is assumed to contain daily log returns. If you decide to use percentage log returns, set `use_percentage_returns = True`. This mainly changes the scale and interpretation of \(\beta^2\).


In [ ]:
SCALE_Y = 100.0

raw_y = pd.read_csv("data.csv", header=None).values.flatten()
y_all = SCALE_Y * raw_y.astype(float)

plt.figure(figsize=(10, 4))
plt.plot(y_all, label="Observed log returns")
plt.title("Observed time series (percentage log returns)")
plt.xlabel("Time")
plt.ylabel("Log return (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Total length:", len(y_all))
print("Mean:", np.mean(y_all))
print("Std:", np.std(y_all))
print("Min, max:", np.min(y_all), np.max(y_all))


## Core PMMH implementation

All log returns are expressed in percentage units, i.e. $y_t \leftarrow 100y_t$. This rescales the posterior of $eta^2$ by $100^2$, while the posterior of $\sigma^2$, $ho$, and the latent path are unchanged by the unit conversion.


The particle filter provides

\[
\widehat p_\theta(y_{1:T})
\]

and one sampled latent trajectory from the particle approximation

\[
X_{1:T}\sim \widehat p(dx_{1:T}\mid y_{1:T},\theta).
\]

The PMMH step accepts \(\theta'\) with probability

\[
1\wedge
\frac{
\widehat p_{\theta'}(y_{1:T})p(\theta')q(\theta(k-1)\mid \theta')
}{
\widehat p_{\theta(k-1)}(y_{1:T})p(\theta(k-1))q(\theta'\mid \theta(k-1))
}.
\]

In this implementation, the random-walk proposal is symmetric on the transformed scale

\[
\eta_1=\log\sigma^2,\qquad
\eta_2=\log\beta^2,\qquad
\eta_3=\operatorname{arctanh}(\rho).
\]

Therefore the log acceptance ratio includes the Jacobian correction

\[
\log J(\theta)=\log\sigma^2+\log\beta^2+\log(1-\rho^2).
\]


In [ ]:
def log_prior(theta):
    """Log prior for theta = (sigma2, beta2, rho)."""
    sigma2, beta2, rho = theta
    if sigma2 <= 0 or beta2 <= 0 or not (-1.0 < rho < 1.0):
        return -np.inf

    lp_sigma = invgamma.logpdf(sigma2, a=1.0, scale=1.0)
    lp_beta = invgamma.logpdf(beta2, a=1.0, scale=1.0)
    lp_rho = -np.log(2.0)  # Uniform(-1, 1)
    return lp_sigma + lp_beta + lp_rho


def theta_to_eta(theta):
    """Map constrained theta to unconstrained eta."""
    sigma2, beta2, rho = theta
    return np.array([np.log(sigma2), np.log(beta2), np.arctanh(rho)])


def eta_to_theta(eta):
    """Map unconstrained eta back to theta."""
    log_sigma2, log_beta2, z_rho = eta
    sigma2 = np.exp(log_sigma2)
    beta2 = np.exp(log_beta2)
    rho = np.tanh(z_rho)
    return np.array([sigma2, beta2, rho])


def log_jacobian(theta):
    """log |d theta / d eta| for eta=(log sigma2, log beta2, atanh rho)."""
    sigma2, beta2, rho = theta
    if sigma2 <= 0 or beta2 <= 0 or not (-1.0 < rho < 1.0):
        return -np.inf
    return np.log(sigma2) + np.log(beta2) + np.log(1.0 - rho**2)


def systematic_resampling(weights, rng):
    """Systematic resampling from normalized weights."""
    N = len(weights)
    positions = (rng.random() + np.arange(N)) / N
    cumulative_sum = np.cumsum(weights)
    cumulative_sum[-1] = 1.0

    indexes = np.zeros(N, dtype=int)
    i, j = 0, 0
    while i < N:
        if positions[i] <= cumulative_sum[j]:
            indexes[i] = j
            i += 1
        else:
            j += 1
    return indexes


def log_g_y_given_x(y_t, x_t, beta2):
    """
    Observation density:
        Y_t | X_t=x_t ~ N(0, beta2 * exp(x_t)).
    """
    return -0.5 * (
        np.log(2.0 * np.pi)
        + np.log(beta2)
        + x_t
        + (y_t**2) * np.exp(-x_t) / beta2
    )


def bootstrap_particle_filter(y, theta, N_particles=500, rng=None, return_path=True):
    """
    SMC step in the PMMH pseudo-code.

    Input:
        theta = (sigma2, beta2, rho)

    Output:
        loglik_hat: log of the particle likelihood estimate
        x_sample: one sampled latent trajectory from the particle approximation
    """
    if rng is None:
        rng = np.random.default_rng()

    y = np.asarray(y)
    T = len(y)
    sigma2, beta2, rho = theta

    if sigma2 <= 0 or beta2 <= 0 or not (-1.0 < rho < 1.0):
        return -np.inf, None

    particles = np.zeros((N_particles, T))
    ancestors = np.zeros((N_particles, T), dtype=int)
    ancestors[:, 0] = -1

    # Initial distribution specified in the coursework: X_0 ~ N(0, 0.01).
    particles[:, 0] = rng.normal(0.0, np.sqrt(0.01), size=N_particles)

    logw = log_g_y_given_x(y[0], particles[:, 0], beta2)
    loglik_hat = logsumexp(logw) - np.log(N_particles)
    weights = np.exp(logw - logsumexp(logw))

    for t in range(1, T):
        parent_idx = systematic_resampling(weights, rng)
        ancestors[:, t] = parent_idx

        particles[:, t] = (
            rho * particles[parent_idx, t - 1]
            + np.sqrt(sigma2) * rng.normal(size=N_particles)
        )

        logw = log_g_y_given_x(y[t], particles[:, t], beta2)
        loglik_hat += logsumexp(logw) - np.log(N_particles)
        weights = np.exp(logw - logsumexp(logw))

    if not return_path:
        return loglik_hat, None

    # Sample one complete trajectory by backward tracing.
    x_sample = np.zeros(T)
    idx = rng.choice(N_particles, p=weights)
    x_sample[-1] = particles[idx, -1]
    for t in range(T - 1, 0, -1):
        idx = ancestors[idx, t]
        x_sample[t - 1] = particles[idx, t - 1]

    return loglik_hat, x_sample


def pmmh_sampler(
    y,
    theta_init=(0.2, 0.2, 0.95),
    n_iter=3000,
    N_particles=200,
    proposal_sd=(0.04, 0.04, 0.03),
    rng=None,
):
    """
    Particle Marginal Metropolis--Hastings following the provided pseudo-code.

    At k=0:
        set theta(0), run SMC, sample X(0), compute p_hat_theta(0)(y).
    At k>=1:
        propose theta_prime, run SMC, sample X_prime, compute p_hat_theta_prime(y),
        accept/reject using the PMMH ratio.
    """
    if rng is None:
        rng = np.random.default_rng()

    y = np.asarray(y)
    T = len(y)

    theta_path = np.zeros((n_iter, 3))
    x_path = np.zeros((n_iter, T))
    loglik_path = np.zeros(n_iter)
    logpost_path = np.zeros(n_iter)
    accepted = np.zeros(n_iter, dtype=bool)

    # k = 0
    theta_curr = np.array(theta_init, dtype=float)
    eta_curr = theta_to_eta(theta_curr)

    loglik_curr, x_curr = bootstrap_particle_filter(
        y, theta_curr, N_particles=N_particles, rng=rng, return_path=True
    )
    logpost_curr = loglik_curr + log_prior(theta_curr) + log_jacobian(theta_curr)

    theta_path[0] = theta_curr
    x_path[0] = x_curr
    loglik_path[0] = loglik_curr
    logpost_path[0] = logpost_curr

    proposal_sd = np.asarray(proposal_sd, dtype=float)

    for k in range(1, n_iter):
        # Proposal theta_prime ~ q(theta_prime | theta(k-1)), implemented on eta-scale.
        eta_prop = eta_curr + rng.normal(0.0, proposal_sd, size=3)
        theta_prop = eta_to_theta(eta_prop)

        logprior_prop = log_prior(theta_prop)
        logjac_prop = log_jacobian(theta_prop)

        if np.isfinite(logprior_prop + logjac_prop):
            # Run SMC targeting p(x_{1:T} | y_{1:T}, theta_prime).
            loglik_prop, x_prop = bootstrap_particle_filter(
                y, theta_prop, N_particles=N_particles, rng=rng, return_path=True
            )

            logpost_prop = loglik_prop + logprior_prop + logjac_prop

            # Since q is symmetric on eta-scale, q cancels there.
            log_alpha = logpost_prop - logpost_curr

            if np.log(rng.uniform()) < min(0.0, log_alpha):
                theta_curr = theta_prop
                eta_curr = eta_prop
                x_curr = x_prop
                loglik_curr = loglik_prop
                logpost_curr = logpost_prop
                accepted[k] = True

        # Otherwise reject and retain previous state.
        theta_path[k] = theta_curr
        x_path[k] = x_curr
        loglik_path[k] = loglik_curr
        logpost_path[k] = logpost_curr

    return {
        "theta_path": theta_path,
        "sigma2_path": theta_path[:, 0],
        "beta2_path": theta_path[:, 1],
        "rho_path": theta_path[:, 2],
        "x_path": x_path,
        "loglik_path": loglik_path,
        "logpost_path": logpost_path,
        "accepted": accepted,
        "accept_rate": np.mean(accepted[1:]),
    }


## Run PMMH

For a first test, use smaller `n_iter` and `N_particles`. For the final report, increase them, for example `n_iter=10000` and `N_particles=500`, if runtime allows.


In [ ]:
rng = np.random.default_rng(123)

T_train = 1000
y_train = y_all[:T_train]

# Posterior-informed starting value, consistent with the PG-Gibbs run and percent scaling.
# These values are only used to initialise PMMH; the stationary distribution is determined by the chain.
theta_init = (0.025, 0.25, 0.96)  # (sigma2, beta2, rho)

out = pmmh_sampler(
    y_train,
    theta_init=theta_init,
    n_iter=10000,
    N_particles=500,
    proposal_sd=(0.08, 0.08, 0.05),  # eta-scale: log sigma2, log beta2, atanh rho
    rng=rng,
)

sigma2_path = out["sigma2_path"]
beta2_path = out["beta2_path"]
rho_path = out["rho_path"]
x_path = out["x_path"]

print("PMMH acceptance rate:", out["accept_rate"])
print("Posterior mean after burn-in will be computed below.")



## Posterior diagnostics

PMMH produces autocorrelated MCMC samples. Trace plots and ACF plots should be interpreted as diagnostics, not as evidence that the draws are independent.


In [ ]:
burn_in = 500

sigma2_post = sigma2_path[burn_in:]
beta2_post = beta2_path[burn_in:]
rho_post = rho_path[burn_in:]
x_post = x_path[burn_in:]

print("Posterior means:")
print("E[sigma2 | y] =", np.mean(sigma2_post))
print("E[beta2  | y] =", np.mean(beta2_post))
print("E[rho    | y] =", np.mean(rho_post))

In [ ]:
def plot_kde(samples, xlabel, title):
    samples = np.asarray(samples)
    grid = np.linspace(np.min(samples), np.max(samples), 300)
    kde = gaussian_kde(samples)
    dens = kde(grid)

    plt.figure(figsize=(6, 4))
    plt.plot(grid, dens)
    plt.fill_between(grid, dens, alpha=0.3)
    plt.axvline(np.mean(samples), linestyle="--", label="Posterior mean")
    plt.xlabel(xlabel)
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_kde(sigma2_post, r"$\sigma^2$", r"Marginal posterior of $\sigma^2$")
plot_kde(beta2_post, r"$\beta^2$", r"Marginal posterior of $\beta^2$")
plot_kde(rho_post, r"$\rho$", r"Marginal posterior of $\rho$")

In [ ]:
def trace_plot(sigma2_path, beta2_path, rho_path):
    fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
    axes[0].plot(sigma2_path)
    axes[0].set_ylabel(r"$\sigma^2$")
    axes[0].set_title("Trace plots")

    axes[1].plot(beta2_path)
    axes[1].set_ylabel(r"$\beta^2$")

    axes[2].plot(rho_path)
    axes[2].set_ylabel(r"$\rho$")
    axes[2].set_xlabel("Iteration")

    plt.tight_layout()
    plt.show()

trace_plot(sigma2_path, beta2_path, rho_path)

In [ ]:
def acf(x, max_lag=100):
    x = np.asarray(x)
    x = x - np.mean(x)
    n = len(x)
    var = np.dot(x, x) / n
    vals = [1.0]
    for lag in range(1, max_lag + 1):
        vals.append(np.dot(x[:-lag], x[lag:]) / ((n - lag) * var))
    return np.array(vals)

max_lag = 150
acf_sigma2 = acf(sigma2_post, max_lag=max_lag)
acf_beta2 = acf(beta2_post, max_lag=max_lag)
acf_rho = acf(rho_post, max_lag=max_lag)

fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
axes[0].stem(range(max_lag + 1), acf_sigma2)
axes[0].set_title(r"ACF of $\sigma^2$")
axes[0].axhline(0.0, linewidth=0.8)

axes[1].stem(range(max_lag + 1), acf_beta2)
axes[1].set_title(r"ACF of $\beta^2$")
axes[1].axhline(0.0, linewidth=0.8)

axes[2].stem(range(max_lag + 1), acf_rho)
axes[2].set_title(r"ACF of $\rho$")
axes[2].axhline(0.0, linewidth=0.8)
axes[2].set_xlabel("Lag")

plt.tight_layout()
plt.show()


## Posterior predictive simulation

For each retained PMMH draw \((\sigma^{2(m)},\beta^{2(m)},\rho^{(m)},X_{1:T}^{(m)})\), start from \(X_T^{(m)}\), simulate future latent states, then simulate future returns.


In [ ]:
def predictive_simulation_sv(sigma2_post, beta2_post, rho_post, x_path_post, H=250, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    M = len(sigma2_post)
    y_future_samples = np.zeros((M, H))
    x_future_samples = np.zeros((M, H))

    for m in range(M):
        sigma2 = sigma2_post[m]
        beta2 = beta2_post[m]
        rho = rho_post[m]
        x_curr = x_path_post[m, -1]

        for h in range(H):
            x_next = rho * x_curr + np.sqrt(sigma2) * rng.normal()
            y_next = np.sqrt(beta2 * np.exp(x_next)) * rng.normal()

            x_future_samples[m, h] = x_next
            y_future_samples[m, h] = y_next
            x_curr = x_next

    return y_future_samples, x_future_samples


H = min(250, len(y_all) - T_train)
y_future_samples, x_future_samples = predictive_simulation_sv(
    sigma2_post,
    beta2_post,
    rho_post,
    x_post,
    H=H,
    rng=rng,
)

pred_mean = np.mean(y_future_samples, axis=0)
lower = np.quantile(y_future_samples, 0.005, axis=0)
upper = np.quantile(y_future_samples, 0.995, axis=0)

future_index = np.arange(T_train, T_train + H)
y_true_future = y_all[T_train:T_train + H]

plt.figure(figsize=(10, 4))
plt.fill_between(future_index, lower, upper, alpha=0.3, label="99% predictive interval")
plt.plot(future_index, pred_mean, label="Predictive mean")
plt.plot(future_index, y_true_future, linestyle="--", label="Observed future")
plt.xlabel("Time index")
plt.ylabel("Future log return (%)")
plt.title("Posterior prediction by PMMH")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: check whether predictive variance stabilizes over the forecast horizon.
pred_var = np.var(y_future_samples, axis=0)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, H + 1), pred_var)
plt.xlabel("Prediction horizon")
plt.ylabel("Empirical predictive variance")
plt.title("Predictive variance over forecast horizon")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()